In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/project")


def read_env(path):
    values = {}
    for number, line in enumerate(path.read_text().splitlines(), 1):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        if not separator or not key or key != key.strip():
            raise ValueError(f"Invalid KEY=value line {number} in {path}")
        values[key] = value
    return values


CONFIG = read_env(PROJECT_DIR / ".colab.env")
R2_CREDENTIALS = read_env(Path("/content/.colab-r2.env"))
for key in ("R2_BUCKET", "R2_ARTIFACT_PREFIX", "EXPECT_GPU"):
    if not CONFIG.get(key):
        raise ValueError(f"Missing {key} in .colab.env")
for key in ("R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY"):
    if not R2_CREDENTIALS.get(key):
        raise ValueError(f"Missing {key} in the external R2 credentials file")
if CONFIG["EXPECT_GPU"] not in ("true", "false"):
    raise ValueError("EXPECT_GPU must be true or false")
ARTIFACT_PREFIX = CONFIG["R2_ARTIFACT_PREFIX"].strip("/")
if not ARTIFACT_PREFIX:
    raise ValueError("R2_ARTIFACT_PREFIX must name a folder")
if CONFIG.get("R2_DATA_PREFIX", "").strip("/") != ARTIFACT_PREFIX:
    raise ValueError("R2_DATA_PREFIX must match R2_ARTIFACT_PREFIX")

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

gpu_command = shutil.which("nvidia-smi")
gpu = (
    subprocess.run([gpu_command, "-L"], capture_output=True, text=True)
    if gpu_command
    else None
)
if CONFIG["EXPECT_GPU"] == "true" and (gpu is None or gpu.returncode != 0):
    raise RuntimeError("GPU expected but nvidia-smi did not find one")
print(f"Python: {sys.version.split()[0]} on {platform.platform()}")
print(f"Working directory: {Path.cwd()}")
print(f"GPU: {gpu.stdout.strip() if gpu and gpu.returncode == 0 else 'none'}")
print(f"R2 bucket: {CONFIG['R2_BUCKET']} | data: {DATA_DIR} | output: {OUTPUT_DIR}")
print(f"Artifact prefix: {ARTIFACT_PREFIX}")
print("R2 credentials: present")

In [ ]:
%pip install -qq --disable-pip-version-check uv boto3

import subprocess

requirements = PROJECT_DIR / "requirements-colab.txt"
subprocess.run(
    [
        "uv",
        "export",
        "--frozen",
        "--no-dev",
        "--extra",
        "experiment",
        "--prune",
        "torch",
        "--prune",
        "numpy",
        "--prune",
        "fsspec",
        "--prune",
        "rich",
        "--prune",
        "colorama",
        "--no-emit-project",
        "--no-hashes",
        "--format",
        "requirements.txt",
        "--output-file",
        str(requirements),
        "--project",
        str(PROJECT_DIR),
    ],
    check=True,
)
%pip install -qq --disable-pip-version-check --requirement {requirements}
%pip install -qq --disable-pip-version-check --no-deps {PROJECT_DIR}

In [ ]:
from jlens_reasoning.environments.colab import source_bundle_sha256

PROJECT_SOURCE_SHA256 = source_bundle_sha256(PROJECT_DIR)
print(f"Project source SHA-256: {PROJECT_SOURCE_SHA256}")

In [ ]:
def r2_client():
    import boto3

    return boto3.client(
        "s3",
        endpoint_url=f"https://{R2_CREDENTIALS['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com",
        aws_access_key_id=R2_CREDENTIALS["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=R2_CREDENTIALS["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )


def upload_artifacts():
    """Upload files in OUTPUT_DIR under the project's artifact prefix."""
    client = r2_client()
    count = 0
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_symlink():
            raise ValueError(f"Refusing to upload symlink: {path}")
        if path.is_file():
            key = f"{ARTIFACT_PREFIX}/{path.relative_to(OUTPUT_DIR).as_posix()}"
            client.upload_file(str(path), CONFIG["R2_BUCKET"], key)
            count += 1
    print(f"Uploaded {count} file(s) from {OUTPUT_DIR}")

# FLenQA: second-order probe J-gain

Can we change final-hidden-state sensitivity while preserving the probe projection, and does that change generated answers? See [the short plan](README.md).

Start with **development**: 12 validation problems, one randomly selected matched rendering per problem, 500 versus 3,000 nominal tokens. Evaluate all eligible test problems only after saving development settings. This existing test split has prior descriptive analysis, so its intervention results are **exploratory**.

Shared derivatives and controls live in `jlens_reasoning`; `experiment.py` only orchestrates FLenQA conditions. No static lens asset, retraining, or old answer rerun is needed.

In [ ]:
PHASE = "development"  # development or evaluation
# After inspecting development diagnostics, set one tested strength here to
# save frozen_settings.json. Leave None while exploring; never tune on test.
FREEZE_STRENGTH = None

settings = {
    "layer": 19,
    "short_ctx": 500,
    "long_ctx": 3000,
    "seed": 1729,
    "relative_strengths": [0.001, 0.003, 0.01],  # delta norm / clean hidden norm
    "control_seeds": [11, 29, 47],
    "fd_relative_steps": [0.001, 0.003],
    "fd_rtol": 0.05,
    "fd_atol": 1e-4,
    "probe_score_atol": 1e-4,
    "probe_score_rtol": 1e-4,
    "dtype": "float32",
    "attention": "eager",
}
if PHASE not in ("development", "evaluation"):
    raise ValueError("Choose development or evaluation")
if PHASE == "evaluation" and FREEZE_STRENGTH is not None:
    raise ValueError("Settings can only be frozen from development")

In [ ]:
from jlens_reasoning.environments.colab import download_r2_inputs

paths = [
    "assets/models/qwen3.5-4b/",
    "datasets/flenqa/",
    "checkpoints/flenqa-probe-assets/",
]
if PHASE == "evaluation":
    paths.append("runs/flenqa-probe-jgain/development/")
download_r2_inputs(
    client=r2_client(),
    bucket=CONFIG["R2_BUCKET"],
    prefix=ARTIFACT_PREFIX,
    destination=DATA_DIR,
    paths=paths,
)

In [ ]:
import hashlib
import json
from dataclasses import asdict

from datasets import load_from_disk

from experiments.flenqa_probe_jgain.experiment import (
    select_pairs,
)
from experiments.flenqa_probe_jlens.analysis import validate_split
from experiments.jlens_readout_sanity.constants import MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows, prepare_prompts
from jlens_reasoning.environments.colab import initialize_colab
from jlens_reasoning.experiments_utils.artifacts import write_results

context = initialize_colab(enable_wandb=False, require_cuda=True)
ASSET_DIR = context.checkpoints_dir / "flenqa-probe-assets"
RESULT_DIR = OUTPUT_DIR / "runs/flenqa-probe-jgain" / PHASE
DEVELOPMENT_DIR = context.runs_dir / "flenqa-probe-jgain/development"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
# Invalidate completion before any partial outputs can replace an older run.
write_results(
    RESULT_DIR / "run_metadata.json",
    {
        "status": "running",
        "phase": PHASE,
        "source_sha256": PROJECT_SOURCE_SHA256,
    },
)


def file_sha256(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

## 1. Bind the run to its inputs

Frozen probes retain their chat-v2 contract. Hash the model files, probe assets, normalized dataset rows, and shared source. Evaluation requires a complete development run and unchanged inputs/code. The notebook source hash is recorded separately because changing phase edits the notebook.

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
rows = normalize_rows(
    dataset["eval"] if hasattr(dataset, "keys") else dataset, full=True
)
split = json.loads((ASSET_DIR / "problem_split.json").read_text())
problem_to_split = validate_split(split, rows)
prompts = prepare_prompts(rows)
prompt_by_id = {prompt.prompt_id: prompt for prompt in prompts}
input_hashes = {
    name: file_sha256(ASSET_DIR / name)
    for name in ("probes.pt", "metadata.json", "problem_split.json")
}
input_hashes["normalized_dataset"] = hashlib.sha256(
    json.dumps([asdict(row) for row in rows], sort_keys=True).encode()
).hexdigest()
input_hashes["model_files"] = {
    path.relative_to(MODEL_PATH).as_posix(): file_sha256(path)
    for path in sorted(Path(MODEL_PATH).rglob("*"))
    if path.is_file()
}
implementation_files = [
    *sorted((PROJECT_DIR / "src").rglob("*.py")),
    PROJECT_DIR / "experiments/flenqa_probe_jgain/experiment.py",
    PROJECT_DIR / "experiments/flenqa_probe_jlens/analysis.py",
    PROJECT_DIR / "experiments/flenqa_probe_jlens/constants.py",
    PROJECT_DIR / "uv.lock",
]
implementation_hashes = {
    str(path.relative_to(PROJECT_DIR)): file_sha256(path)
    for path in implementation_files
}

In [ ]:
frozen = None
if PHASE == "evaluation":
    frozen = json.loads((DEVELOPMENT_DIR / "frozen_settings.json").read_text())
    development = json.loads((DEVELOPMENT_DIR / "run_metadata.json").read_text())
    if development["status"] != "complete":
        raise ValueError("Development run is incomplete")
    for name, digest in frozen["development_hashes"].items():
        if file_sha256(DEVELOPMENT_DIR / name) != digest:
            raise ValueError(f"Development artifact changed: {name}")
    if (
        frozen["input_hashes"] != input_hashes
        or frozen["implementation_hashes"] != implementation_hashes
    ):
        raise ValueError("Inputs or implementation changed since settings were frozen")
    settings = frozen["settings"]
display(settings)

## 2. Select matched problems before observing outcomes

Development uses two problems per task/label stratum from validation. Evaluation uses all eligible test problems. Choose one matched padding/dispersion condition per problem using the fixed seed. Coverage includes every problem in the requested partition, including missing pairs. No selection uses answer correctness or probe strength.

In [ ]:
selected, coverage = select_pairs(
    prompts,
    problem_to_split,
    phase=PHASE,
    seed=settings["seed"],
    short_ctx=settings["short_ctx"],
    long_ctx=settings["long_ctx"],
)
selected_ids = sorted(set(selected.short_id) | set(selected.long_id))
selected.to_parquet(RESULT_DIR / "selected_pairs.parquet", index=False)
coverage.to_parquet(RESULT_DIR / "coverage.parquet", index=False)
display(selected[["problem_id", "task", "label", "padding_type", "dispersion"]])
display(coverage.groupby("reason").size().rename("problems"))

In [ ]:
upload_artifacts()
print("COLAB_NOTEBOOK_UPLOAD_COMPLETE")